# Three-body free fall — adaptive RK4, base V1.5

Run the cells in order in Google Colab. The notebook is self-contained: it does not need the local Python script, your Google Drive, an account-specific path, or a manual data upload. Each reader runs a separate session.

The equations use **G = 1**, three unit masses, positions **(x, y), (-0.5, 0), (0.5, 0)** and zero initial velocities. Integration and the four proximity calculations use **C++ with Boost.Multiprecision** by default, at the precision selected by `dps`. Python retains the controls, plots and animations; only display copies use float64. Set `INTEGRATION_BACKEND = "python"` to use the original mpmath solver for comparison.

## Data and attribution
The default data source is the authors' [public catalogue](https://db2.fmi.uni-sofia.bg/3bodyfree/) and its [sol_80.txt](https://db2.fmi.uni-sofia.bg/3bodyfree/sol_80.txt) download: 24,582 initial conditions, with columns `x y T T*` ordered by `T*`.

Ivan Hristov, Radoslava Hristova, Veljko Dmitrašinović and Kiyotaka Tanikawa, *Three-body periodic collisionless equal-mass free-fall orbits revisited*, [DOI: 10.1007/s10569-023-10177-w](https://doi.org/10.1007/s10569-023-10177-w), [arXiv:2308.16159](https://arxiv.org/abs/2308.16159).

The catalogue is fetched into memory and checked against the version used to prepare this notebook. No simulation results, checkpoints or movies are written to disk. Only temporary C++ source/executable build files are created in the session; compilation is repeated after the runtime is replaced.

## 1. Enable interactive plots
Run this cell once after connecting to a runtime. Interactive plots use [ipympl's Colab support](https://matplotlib.org/ipympl/installing.html).

If the first cell reports an invalid backend after installation, choose **Runtime → Restart session**, then rerun this cell before starting integration. Do not restart a session while an integration you want to keep is running. Successful setup prints `Interactive backend: module://ipympl.backend_nbagg`.


In [ ]:
# Install missing packages without importing the interactive backend.
import importlib.util
import subprocess
import sys

required_packages = ("mpmath", "numpy", "matplotlib", "ipympl")
missing_packages = [
    name for name in required_packages
    if importlib.util.find_spec(name) is None
]
_RK4_WIDGET_RESTART_REQUIRED = globals().get("_RK4_WIDGET_RESTART_REQUIRED", False)
if missing_packages:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", *missing_packages
    ])
    # A newly installed widget backend may not be visible to this kernel.
    _RK4_WIDGET_RESTART_REQUIRED = True

if _RK4_WIDGET_RESTART_REQUIRED:
    print("Installation complete. Select Runtime > Restart session once, "
          "then run all cells from the top. No installation is repeated "
          "when the required packages are already available.")
else:
    print("Dependencies ready. Continue to the interactive display setup.")


### Activate the interactive display
If installation above requested a restart, select **Runtime > Restart session** once, then run the notebook from the top. This is needed only when packages have just been installed in the current runtime. Do this before starting the integration.


In [ ]:
# Activate widgets only after the installation session has been restarted.
if globals().get("_RK4_WIDGET_RESTART_REQUIRED", False):
    raise RuntimeError(
        "Dependencies were installed in this session. Select Runtime > "
        "Restart session, then run all cells from the top."
    )

from google.colab import output
output.enable_custom_widget_manager()

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except (ImportError, ValueError, RuntimeError) as error:
    raise RuntimeError(
        "The widget backend could not start. Select Runtime > Restart "
        "session, then run all cells from the top."
    ) from error

import matplotlib.pyplot as plt
print("Interactive backend:", plt.get_backend())


## 2. Choose the initial condition and numerical settings

`ic_number` is a **1-based catalogue row number**. `period_count` controls the total integration time: `period_count * T`. To supply your own coordinates and reference period, uncomment the complete `MANUAL_IC` block. Keep decimal values in quotes. Manual mode does not download the catalogue.

The precision and tolerances below match the desktop script. These settings can require long CPU runs; Colab session limits still apply. Lower `save_every` retains more points for the display without changing adaptive integration.


In [ ]:
from pathlib import Path
from time import perf_counter

import mpmath as mp


# =============================
# Hyperparameters – edit here
# =============================
INTEGRATION_BACKEND = "cpp"  # <<< SELECT "cpp" OR THE ORIGINAL "python" BACKEND. >>>

SOL80_TXT = None  # The public catalogue is loaded into memory in the next cells.

HYPERPARAMETERS = {
    "ic_number": 15,  # <<< SELECT IC HERE: row number in sol_80.txt, starting at 1. >>>
    "period_count": 1,  # <<< SELECT THE NUMBER OF PERIODS HERE: end time = period_count * T. >>>
    "dps": 50,
    "h0": "1e-3",  # Initial step size; subsequent steps are chosen adaptively.
    "rtol": "1e-15",
    "atol": "1e-15",
    "periodicity_threshold": "0.1",
    "safety": "0.9",
    "min_factor": "0.2",
    "max_factor": "2.0",
    "max_steps": 35_000_000,
    "progress_every": 10_000,  # <<< PRINT PROGRESS EVERY N ACCEPTED STEPS. >>>
    "save_every": 3_000,  # <<< KEEP A PLOT SAMPLE IN MEMORY EVERY N ACCEPTED STEPS. >>>
    "view_tilt_degrees": 15,  # <<< VIEW TILT FROM DIRECTLY ABOVE: 0–90 DEGREES. >>>
    "plot_linewidth": 1.26,  # Static orbit line width in points (10% thinner).
    "animation_linewidth": 1.8,  # Animated orbit trail width in points (10% thinner).
    "triangle_linewidth": 2.25,  # Triangle side width in points (10% thinner).
    "animate": True,  # <<< SHOW THE ANIMATION AFTER INTEGRATION. >>>
    "animation_mode": "triangle",  # <<< DISPLAY MODE: "triangle" (coloured sides) or "orbits" (trajectories). >>>
    "animation_renderer": "browser",  # <<< "browser" FOR SMOOTH COLAB PLAYBACK; "matplotlib" FOR THE LEGACY VIEW. >>>
    "animation_max_points": 1500,  # Maximum trail vertices per body/frame (display only).
    "animation_fps": 30,  # Animation frame rate.
    "animation_seconds": 30,  # Playback seconds for the complete trajectory at 1x speed.
    "animation_speed": 1.0,  # Initial speed; the slider ranges from 0 (paused) to 5x.
    "animation_trail_fraction": 0.15,  # Initial trail fraction: 0 = bodies only, 1 = full elapsed trail.
}



# Manually choose the position and integration time for body one.

In [ ]:
# Optional manual input. Uncomment the complete block below to override ic_number.
# Keep decimal values in quotes so they are preserved through the decimal-text C++ interface.
# T is the reference period; total integration time is period_count * T.
MANUAL_IC = None
# MANUAL_IC = {
#     "x": "0.25694268581619406691269277542214527477259843827894882545958420075540861203009715",
#     "y": "0.2783746621563187640173069301772599105064839871107602666586336860267930150563534",
#     "T": "0.21573615905021502937878732480336592981156797723875640215934585591298459904348956e1",
# }

## 3. Public catalogue loader

No Google Drive authorization is needed. The download happens only when catalogue mode is used.


In [ ]:
from functools import lru_cache
from hashlib import sha256
from urllib.request import urlopen

CATALOGUE_URL = "https://db2.fmi.uni-sofia.bg/3bodyfree/sol_80.txt"
CATALOGUE_SHA256 = "81dc1b70828fd5960dec57f64ac88f7945fa5c8eddf1cfb6ea33b3475c55373c"

@lru_cache(maxsize=1)
def download_catalogue():
    # Fetch decimal text once per session. Do not convert the catalogue to float64.
    print(f"Loading the authors' public catalogue: {CATALOGUE_URL}", flush=True)
    with urlopen(CATALOGUE_URL, timeout=60) as response:
        data = response.read()
    if sha256(data).hexdigest() != CATALOGUE_SHA256:
        raise ValueError("The catalogue has changed. Verify the source before updating its checksum.")
    print(f"Catalogue verified: {len(data.splitlines())} initial conditions.")
    return data.decode("utf-8")


## 4. Integrator and display functions

Run this definition cell once, then run the integration cell below.


In [ ]:
def validate_hyperparameters(settings):
    for key in ("ic_number", "period_count", "dps", "max_steps",
                "progress_every", "save_every"):
        if type(settings[key]) is not int or settings[key] < 1:
            raise ValueError(f"{key} must be a positive integer.")
    with mp.workdps(settings["dps"]):
        for key in ("h0", "rtol", "atol", "periodicity_threshold",
                    "safety", "min_factor", "max_factor"):
            value = mp.mpf(settings[key])
            if not mp.isfinite(value) or value <= 0:
                raise ValueError(f"{key} must be finite and positive.")
        if not mp.mpf(settings["safety"]) < 1:
            raise ValueError("safety must be less than 1.")
        if not mp.mpf(settings["min_factor"]) < 1 < mp.mpf(settings["max_factor"]):
            raise ValueError("Requires min_factor < 1 < max_factor.")
    if not 0 <= settings["view_tilt_degrees"] <= 90:
        raise ValueError("view_tilt_degrees must be between 0 and 90.")
    for key in ("plot_linewidth", "animation_linewidth", "triangle_linewidth"):
        if not mp.isfinite(settings[key]) or settings[key] <= 0:
            raise ValueError(f"{key} must be finite and positive.")
    if settings.get("animation_renderer", "browser") not in {"browser", "matplotlib"}:
        raise ValueError('animation_renderer must be "browser" or "matplotlib".')
    if type(settings.get("animation_max_points", 1500)) is not int or settings.get("animation_max_points", 1500) < 4:
        raise ValueError('animation_max_points must be an integer of at least 4.')
    if type(settings["animate"]) is not bool:
        raise ValueError("animate must be True or False.")
    if settings["animation_mode"] not in {"triangle", "orbits"}:
        raise ValueError('animation_mode must be "triangle" or "orbits".')
    for key in ("animation_fps", "animation_seconds"):
        value = mp.mpf(settings[key])
        if not mp.isfinite(value) or value <= 0:
            raise ValueError(f"{key} must be finite and positive.")
    if not 0 <= settings["animation_speed"] <= 5:
        raise ValueError("animation_speed must be between 0 and 5.")
    if not 0 <= settings["animation_trail_fraction"] <= 1:
        raise ValueError("animation_trail_fraction must be between 0 and 1.")


def load_initial_condition(path, ic_number):
    """Read the public catalogue in memory; no Drive access or upload is required."""
    return parse_initial_condition(download_catalogue(), ic_number)


def parse_initial_condition(source_text, ic_number):
    """Parse a row from local or downloaded source text at the current precision."""
    lines = source_text.splitlines()
    if not 1 <= ic_number <= len(lines):
        raise ValueError(f"The IC number must be between 1 and {len(lines)}.")
    columns = lines[ic_number - 1].split()
    if len(columns) != 4:
        raise ValueError(f"Row {ic_number} must contain x y T T*.")
    x, y, period = [mp.mpf(value) for value in columns[:3]]
    if not all(mp.isfinite(value) for value in (x, y, period)) or period <= 0:
        raise ValueError("x, y and T must be finite, and T must be positive.")
    return x, y, period


def get_initial_condition(settings):
    """Use the catalogue row unless the optional manual block is enabled."""
    if MANUAL_IC is None:
        values = load_initial_condition(SOL80_TXT, settings["ic_number"])
        return (*values, f"IC {settings['ic_number']}")
    if set(MANUAL_IC) != {"x", "y", "T"} or any(
            not isinstance(value, str) for value in MANUAL_IC.values()):
        raise ValueError('MANUAL_IC must contain x, y and T as decimal strings.')
    x, y, period = [mp.mpf(MANUAL_IC[key]) for key in ("x", "y", "T")]
    if not all(mp.isfinite(value) for value in (x, y, period)) or period <= 0:
        raise ValueError("Manual x, y and T must be finite, and T must be positive.")
    return x, y, period, "Manual initial condition"


# =============================
# Initial state and equations of motion
# =============================
def initial_conditions_mp(x, y):
    # G = 1, three unit masses and zero initial velocities (free fall).
    # State: [x1, y1, x2, y2, x3, y3, vx1, vy1, vx2, vy2, vx3, vy3].
    zero = mp.mpf("0")
    return [x, y, mp.mpf("-0.5"), zero, mp.mpf("0.5"), zero] + [zero] * 6


def vec_sub(a, b):
    return [a[i] - b[i] for i in range(2)]


def vec_add(a, b):
    return [a[i] + b[i] for i in range(2)]


def vec_norm(a):
    return mp.sqrt(sum(ai**2 for ai in a))


def derivatives_mp(t, state):
    r1, r2, r3 = state[0:2], state[2:4], state[4:6]

    def grav_acc(r_from, r_to):
        diff = vec_sub(r_to, r_from)
        dist = vec_norm(diff)
        return [diff[i] / dist**3 for i in range(2)]

    a1 = vec_add(grav_acc(r1, r2), grav_acc(r1, r3))
    a2 = vec_add(grav_acc(r2, r1), grav_acc(r2, r3))
    a3 = vec_add(grav_acc(r3, r1), grav_acc(r3, r2))
    return state[6:12] + a1 + a2 + a3


# =============================
# RK4 with adaptive step size
# =============================
def rk4_step_mp(f, t, y, h):
    f1 = f(t, y)
    f2 = f(t + h/2, [y[i] + h * f1[i] / 2 for i in range(len(y))])
    f3 = f(t + h/2, [y[i] + h * f2[i] / 2 for i in range(len(y))])
    f4 = f(t + h, [y[i] + h * f3[i] for i in range(len(y))])
    return [
        y[i] + h * (f1[i] + 2*f2[i] + 2*f3[i] + f4[i]) / 6
        for i in range(len(y))
    ]


def solve_adaptive_rk4_mp(f, y0, T, *, rtol, atol, h0, safety,
                          min_factor, max_factor, max_steps,
                          progress_every, save_every):
    t = mp.mpf("0")
    h = mp.mpf(h0)
    y = list(y0)
    steps = 0
    rejected = 0
    stop_reason = "completed"
    # Plot sampling does not affect adaptive integration.
    t_values = [t]
    y_values = [list(y)]

    while t < T:
        if steps >= max_steps:
            stop_reason = "max_steps"
            print(f"Stopped at t = {mp.nstr(t, 12)}: step limit {max_steps} reached.", flush=True)
            break

        # Only shorten the final step to reach T; no fixed integration subintervals.
        reaches_end = t + h >= T
        if reaches_end:
            h = T - t
        if t + h == t or t + h/2 == t:
            raise ArithmeticError("The step size is too small for the selected dps.")

        y_big = rk4_step_mp(f, t, y, h)
        y_half = rk4_step_mp(f, t, y, h / 2)
        y_small = rk4_step_mp(f, t + h / 2, y_half, h / 2)

        # Use the original local error control solely to choose the step size.
        err = max(
            abs(y_small[i] - y_big[i])
            / (atol + rtol * max(abs(y[i]), abs(y_small[i])))
            for i in range(len(y))
        )
        if not mp.isfinite(err):
            raise ArithmeticError("The RK4 step produced a non-finite error estimate.")

        if err <= 1:
            t = T if reaches_end else t + h
            y = y_small
            steps += 1

            if steps % save_every == 0:
                t_values.append(t)
                y_values.append(list(y))

            if steps % progress_every == 0:
                print(
                    f"step {steps}, "
                    f"t = {mp.nstr(t, 12)} / {mp.nstr(T, 12)}, "
                    f"{mp.nstr(100 * t / T, 6)} %, "
                    f"h = {mp.nstr(h, 5)}, "
                    f"err = {mp.nstr(err, 5)}, "
                    f"rejected = {rejected}",
                    flush=True,
                )
        else:
            rejected += 1

        if err == 0:
            factor = max_factor
        else:
            factor = safety * err**(-mp.mpf("1") / 5)
            factor = min(max_factor, max(min_factor, factor))
        h *= factor

    # Always include the initial and final points, even at the step limit.
    if t_values[-1] != t:
        t_values.append(t)
        y_values.append(list(y))
    return {
        "t": t,
        "y": y,
        "steps": steps,
        "rejected": rejected,
        "stop_reason": stop_reason,
        "t_values": t_values,
        "y_values": y_values,
    }


# =============================
# Proximity functions – evaluated only at the final point
# =============================
def pairwise_distances_mp(state):
    r1, r2, r3 = state[0:2], state[2:4], state[4:6]
    return sorted([vec_norm(vec_sub(r1, r2)),
                   vec_norm(vec_sub(r1, r3)),
                   vec_norm(vec_sub(r2, r3))])


def proximity_values_mp(state, initial_state):
    return {
        "shape_error": mp.sqrt(sum(
            (a - b)**2 for a, b in zip(pairwise_distances_mp(state),
                                      pairwise_distances_mp(initial_state)))),
        "position_error": vec_norm([state[i] - initial_state[i] for i in range(6)]),
        "velocity_error": vec_norm([state[i] - initial_state[i] for i in range(6, 12)]),
        "state_error": vec_norm([state[i] - initial_state[i] for i in range(12)]),
    }


NEON_COLORS = ("#BD72FF", "#6055FF", "#FF4DA8")


def apply_neon_style(fig):
    """Apply the black background and neon palette without changing orbit data."""
    fig.set_facecolor("black")
    for text in fig.texts:
        text.set_color("white")
    for ax in fig.axes:
        ax.set_facecolor("black")
        ax.tick_params(colors="white")
        for text in [ax.title, ax.xaxis.label, ax.yaxis.label, *ax.texts]:
            text.set_color("white")
        for spine in ax.spines.values():
            spine.set_color("#666666")
        if hasattr(ax, "zaxis"):
            ax.zaxis.label.set_color("white")
            ax.tick_params(axis="z", colors="white")
            for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
                axis.set_pane_color((0, 0, 0, 1))
                axis.line.set_color("#666666")
                axis._axinfo["grid"]["color"] = (0.25, 0.25, 0.25, 0.5)
        legend = ax.get_legend()
        if legend is not None:
            legend.get_frame().set_facecolor("#111111")
            legend.get_frame().set_edgecolor("#666666")
            for text in legend.get_texts():
                text.set_color("white")


def configure_orbit_view(ax, positions, settings):
    """Use equal x/y length scales and the selected camera tilt for planar motion."""
    import numpy as np

    low = positions.min(axis=(0, 1))
    high = positions.max(axis=(0, 1))
    center = (low + high) / 2
    extent = max(float(np.max(high - low)), 1e-12) * 0.6
    ax.set_xlim(center[0] - extent, center[0] + extent)
    ax.set_ylim(center[1] - extent, center[1] + extent)
    ax.set_zlim(-0.2 * extent, 0.2 * extent)
    ax.set_box_aspect((1, 1, 0.2))
    ax.view_init(elev=90 - settings["view_tilt_degrees"], azim=-90)
    ax.set(xlabel="x", ylabel="y")
    ax.set_zticks([])
    ax.zaxis.line.set_visible(False)
    ax.grid(True, alpha=0.2)


def add_periodicity_box(fig, result, settings):
    """Show the final return test, threshold and state error (two significant figures)."""
    values = result.get("proximity_values")
    if values is None:
        values = proximity_values_mp(result["y_values"][-1], result["y_values"][0])
    threshold = result.get("periodicity_threshold", mp.mpf(settings["periodicity_threshold"]))
    if result["stop_reason"] != "completed":
        classification = "not assessed (incomplete)"
    else:
        classification = "periodic" if all(value < threshold for value in values.values()) else "non-periodic"
    text = (
        f"Periodicity (all four): {classification}\n"
        f"Threshold: {mp.nstr(threshold, mp.mp.dps)}\n"
        f"Final state_error: {mp.nstr(values['state_error'], 2, strip_zeros=False)}"
    )
    return fig.text(
        0.04, 0.90, text, ha="left", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#111111", edgecolor="#666666", alpha=0.95),
    )


def create_orbit_plot(result, settings):
    """Create an in-memory orbit plot with body colours and start/end markers."""
    import numpy as np
    import matplotlib.pyplot as plt

    positions = np.array([
        [float(v) for v in state[:6]] for state in result["y_values"]
    ]).reshape(-1, 3, 2)
    if not np.all(np.isfinite(positions)):
        raise ValueError("Trajectory data cannot be displayed as finite float64 values.")
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection="3d")
    fig.subplots_adjust(bottom=0.1, top=0.79)
    configure_orbit_view(ax, positions, settings)
    add_periodicity_box(fig, result, settings)
    for body, color in enumerate(NEON_COLORS):
        path = positions[:, body, :]
        ax.plot(path[:, 0], path[:, 1], np.zeros(len(path)), color=color,
                linewidth=settings["plot_linewidth"], label=f"Body {body + 1}")
        ax.plot([path[0, 0]], [path[0, 1]], [0], "o", color=color, markersize=8)
        ax.plot([path[-1, 0]], [path[-1, 1]], [0], "x", color=color, markersize=8)
    status = "" if result["stop_reason"] == "completed" else " · incomplete integration"
    label = result.get("ic_label", f"IC {settings['ic_number']}")
    fig.suptitle(f"Orbits · {label}{status}", fontsize=15)
    ax.set_title(f"0 ≤ t ≤ {mp.nstr(result['t'], 8)} · tilt {settings['view_tilt_degrees']:g}°")
    ax.legend(loc="upper right")
    fig.text(0.5, 0.05, "○ Start  ·  × End  ·  motion in the plane z = 0", ha="center")
    apply_neon_style(fig)
    return fig


def show_orbit_plot(result, settings):
    import matplotlib.pyplot as plt

    print("Displaying the orbit plot.")
    fig = create_orbit_plot(result, settings)
    plt.show(block=True)
    return fig


class OrbitAnimation:
    """Float64 display copies; mpmath results and classifications are never changed."""

    def __init__(self, result, settings):
        import numpy as np
        import matplotlib.pyplot as plt
        from matplotlib.animation import FuncAnimation
        from matplotlib.widgets import Button, RadioButtons, Slider

        self.np = np
        self.settings_max_points = settings.get("animation_max_points", 1500)
        self.times = np.array([float(t) for t in result["t_values"]])
        self.positions = np.array([
            [float(v) for v in state[:6]] for state in result["y_values"]
        ]).reshape(-1, 3, 2)
        if not np.all(np.isfinite(self.times)) or not np.all(np.isfinite(self.positions)):
            raise ValueError("Trajectory data cannot be displayed as finite float64 values.")
        if len(self.times) < 2 or self.times[-1] <= self.times[0]:
            raise ValueError("The animation requires at least two distinct times.")
        # Very close mpmath times may round to the same float64 time.
        keep = np.r_[np.diff(self.times) > 0, True]
        self.times, self.positions = self.times[keep], self.positions[keep]
        self.duration = self.times[-1] - self.times[0]
        self.simulation_rate = self.duration / settings["animation_seconds"]
        self.speed = settings["animation_speed"]
        self.mode = settings["animation_mode"]
        self.trail_fraction = settings["animation_trail_fraction"]
        self.trail_duration = self.trail_fraction * self.duration
        self.current_time = self.times[0]
        self.last_tick = perf_counter()

        self.fig = plt.figure(figsize=(8, 9))
        self.ax = self.fig.add_subplot(111, projection="3d")
        self.fig.subplots_adjust(bottom=0.35, top=0.79)
        label = result.get("ic_label", f"IC {settings['ic_number']}")
        self.title_prefix = f"Three bodies · {label}"
        configure_orbit_view(self.ax, self.positions, settings)
        self.periodicity_box = add_periodicity_box(self.fig, result, settings)
        if result["stop_reason"] != "completed":
            self.title_prefix += " · incomplete integration"
        self.fig.suptitle(f"{self.title_prefix} · tilt {settings['view_tilt_degrees']:g}°", fontsize=15)

        colors = NEON_COLORS
        self.trails, self.points = [], []
        for body, color in enumerate(colors):
            trail, = self.ax.plot([], [], [], color=color, linewidth=settings["animation_linewidth"])
            point, = self.ax.plot([], [], [], "o", color=color, markersize=9,
                                  markeredgecolor="white", label=f"Body {body + 1}")
            self.trails.append(trail)
            self.points.append(point)
        self.triangle_sides = []
        self.pairs = ((0, 1), (1, 2), (2, 0))
        for (first, second), color in zip(self.pairs, colors):
            side, = self.ax.plot([], [], [], color=color, linewidth=settings["triangle_linewidth"],
                                 label=f"Side {first + 1}–{second + 1}")
            self.triangle_sides.append(side)
        self.legend = None
        self.time_label = self.ax.set_title("")
        self.slider = Slider(
            self.fig.add_axes([0.25, 0.22, 0.55, 0.025]), "Speed",
            0, 5, valinit=self.speed, valfmt="%1.2fx",
        )
        self.tilt_slider = Slider(
            self.fig.add_axes([0.25, 0.165, 0.55, 0.025]), "Tilt from z",
            0, 90, valinit=settings["view_tilt_degrees"], valfmt="%1.1f°",
        )
        self.trail_slider = Slider(
            self.fig.add_axes([0.25, 0.11, 0.55, 0.025]), "Trail length",
            0, 1, valinit=self.trail_fraction, valfmt="%1.2f",
        )
        self.mode_control = RadioButtons(
            self.fig.add_axes([0.15, 0.015, 0.32, 0.055]), ("Orbits", "Triangle"),
            active=1 if self.mode == "triangle" else 0,
        )
        self.restart_button = Button(self.fig.add_axes([0.59, 0.02, 0.22, 0.04]), "Restart", color="#222222", hovercolor="#444444")
        self.fig.text(0.5, 0.285, "0x = paused  ·  1x = full trajectory in "
                      f"{settings['animation_seconds']:g} s", ha="center", fontsize=10)
        self.fig.text(0.5, 0.083, "Orbits: 0 = bodies only · 1 = full elapsed trail", ha="center", fontsize=9)
        self.slider.on_changed(self.set_speed)
        self.tilt_slider.on_changed(self.set_tilt)
        self.trail_slider.on_changed(self.set_trail_fraction)
        self.restart_button.on_clicked(self.restart)
        self.mode_control.on_clicked(self.set_mode)
        apply_neon_style(self.fig)
        self.set_mode("Triangle" if self.mode == "triangle" else "Orbits")
        self.draw()
        # Keep animation and controls as attributes to prevent garbage collection.
        self.animation = FuncAnimation(
            self.fig, self.update, init_func=self.draw, frames=None,
            interval=1000 / settings["animation_fps"], blit=False,
            cache_frame_data=False,
        )

    def position_at(self, t):
        """Linearly interpolate display positions between saved samples."""
        np = self.np
        index = int(np.clip(np.searchsorted(self.times, t, side="right") - 1,
                            0, len(self.times) - 2))
        fraction = np.clip((t - self.times[index])
                           / (self.times[index + 1] - self.times[index]), 0, 1)
        return ((1 - fraction) * self.positions[index]
                + fraction * self.positions[index + 1])

    def draw(self):
        np = self.np
        t = self.current_time
        current = self.position_at(t)
        show_trail = self.mode == "orbits" and self.trail_fraction > 0
        if show_trail:
            start_time = max(self.times[0], t - self.trail_duration)
            left = np.searchsorted(self.times, start_time, side="right")
            right = np.searchsorted(self.times, t, side="left")
            budget = self.settings_max_points - 2
            indices = (np.arange(left, right) if right-left <= budget
                       else np.linspace(left, right-1, budget, dtype=int))
            trail = np.concatenate((self.position_at(start_time)[None, :, :],
                                    self.positions[indices], current[None, :, :]))
            zeros = np.zeros(len(trail))
            for body, line in enumerate(self.trails):
                line.set_data_3d(trail[:, body, 0], trail[:, body, 1], zeros)
        for body, (line, point) in enumerate(zip(self.trails, self.points)):
            line.set_visible(show_trail)
            point.set_data_3d([current[body, 0]], [current[body, 1]], [0])
        self.time_label.set_text(f"t = {t:.6g} / {self.times[-1]:.6g}")
        if self.mode == "triangle":
            for side, (first, second) in zip(self.triangle_sides, self.pairs):
                side.set_data_3d([current[first, 0], current[second, 0]],
                                 [current[first, 1], current[second, 1]], [0, 0])
        return (*self.trails, *self.points, *self.triangle_sides, self.time_label)

    def set_mode(self, label):
        self.mode = "triangle" if label == "Triangle" else "orbits"
        triangle = self.mode == "triangle"
        for line in self.trails:
            line.set_visible(not triangle and self.trail_fraction > 0)
        for side in self.triangle_sides:
            side.set_visible(triangle)
        if self.legend is not None:
            self.legend.remove()
        self.legend = self.ax.legend(
            handles=self.triangle_sides if triangle else self.points,
            loc="upper right",
        )
        apply_neon_style(self.fig)
        self.fig.canvas.draw_idle()

    def advance(self, elapsed):
        if self.speed > 0:
            # Show the final point for one frame before restarting playback.
            if self.current_time >= self.times[-1]:
                self.current_time = self.times[0]
            else:
                self.current_time = min(self.times[-1], self.current_time
                                        + max(0, elapsed) * self.simulation_rate * self.speed)
        return self.draw()

    def update(self, frame):
        now = perf_counter()
        elapsed = now - self.last_tick
        self.last_tick = now
        return self.advance(elapsed)

    def set_speed(self, value):
        self.update(None)
        self.speed = float(value)
        self.fig.canvas.draw_idle()

    def set_tilt(self, value):
        """Change the viewing angle from the z-axis without changing the motion."""
        self.ax.view_init(elev=90 - float(value), azim=self.ax.azim)
        self.fig._suptitle.set_text(f"{self.title_prefix} · tilt {float(value):g}°")
        self.fig.canvas.draw_idle()

    def set_trail_fraction(self, value):
        """Update the visible time window; never draw future trajectory samples."""
        self.trail_fraction = float(value)
        self.trail_duration = self.trail_fraction * self.duration
        self.draw()
        self.fig.canvas.draw_idle()

    def restart(self, event):
        self.current_time = self.times[0]
        self.last_tick = perf_counter()
        self.draw()
        self.fig.canvas.draw_idle()


def show_animation(result, settings):
    import matplotlib.pyplot as plt

    if len(result["t_values"]) < 2:
        print("No animation: too few saved time samples.")
        show_orbit_plot(result, settings)
        return
    if settings.get("animation_renderer", "browser") == "browser":
        from IPython.display import display
        with plt.ioff():
            orbit_fig = create_orbit_plot(result, settings)
        display(orbit_fig.canvas)
        view = create_browser_animation(result, settings)
        display(view)
        return view
    print("Displaying the orbit plot and animation. The speed slider pauses at 0x.")
    print("Select Orbits or Triangle. The triangle sides have three different colours.")
    print("Use Tilt from z to change the view, and Trail length to adjust orbit persistence (0–1).")
    print("Plot detail is controlled by save_every; smaller values retain more samples.")
    orbit_fig = create_orbit_plot(result, settings)
    view = OrbitAnimation(result, settings)
    plt.show(block=True)
    return view


def main(*, show_plots=True):
    validate_hyperparameters(HYPERPARAMETERS)
    mp.mp.dps = HYPERPARAMETERS["dps"]
    x, y, T, label = get_initial_condition(HYPERPARAMETERS)
    period_count = HYPERPARAMETERS["period_count"]
    end_time = period_count * T
    initial_state = initial_conditions_mp(x, y)
    threshold = mp.mpf(HYPERPARAMETERS["periodicity_threshold"])

    print(label)
    print(f"x = {mp.nstr(x, mp.mp.dps)}, y = {mp.nstr(y, mp.mp.dps)}")
    print(f"Number of periods: {period_count}, T = {mp.nstr(T, mp.mp.dps)}")
    print(f"Integrating from 0 to {mp.nstr(end_time, mp.mp.dps)}", flush=True)

    start = perf_counter()
    if INTEGRATION_BACKEND not in {"cpp", "python"}:
        raise ValueError('INTEGRATION_BACKEND must be "cpp" or "python".')
    print(f"Integration backend: {INTEGRATION_BACKEND}", flush=True)
    integrator = solve_adaptive_rk4_cpp if INTEGRATION_BACKEND == "cpp" else solve_adaptive_rk4_mp
    result = integrator(
        derivatives_mp, initial_state, end_time,
        **{key: mp.mpf(HYPERPARAMETERS[key]) for key in
           ("rtol", "atol", "h0", "safety", "min_factor", "max_factor")},
        max_steps=HYPERPARAMETERS["max_steps"],
        progress_every=HYPERPARAMETERS["progress_every"],
        save_every=HYPERPARAMETERS["save_every"],
    )
    print(f"Accepted steps: {result['steps']}")
    result["ic_label"] = label
    print(f"Rejected steps: {result['rejected']}")
    elapsed = result.get("integration_seconds", perf_counter() - start)
    print(f"Elapsed time (integration): {elapsed:.2f} s")
    t, state, stop_reason = result["t"], result["y"], result["stop_reason"]

    values = result.get("proximity_values")
    if values is None:
        values = proximity_values_mp(state, initial_state)
    result["proximity_values"] = values
    result["periodicity_threshold"] = threshold
    print(f"Final point: t = {mp.nstr(t, mp.mp.dps)}")
    for name, value in values.items():
        print(f"  {name}: {mp.nstr(value, mp.mp.dps)}")
    print(f"Periodicity threshold: {mp.nstr(threshold, mp.mp.dps)}")

    if stop_reason != "completed":
        print("Incomplete integration: the step limit was reached before the end time. Periodicity was not classified.")
    else:
        # Use the same two numerical classifications as in rk4_mpmath.py.
        class_1 = "periodic" if values["state_error"] < threshold else "non-periodic"
        class_2 = "periodic" if all(v < threshold for v in values.values()) else "non-periodic"
        print(f"Class 1 (state_error): {class_1}")
        print(f"Class 2 (all four): {class_2}")
    if show_plots:
        if HYPERPARAMETERS["animate"]:
            result["animation_view"] = show_animation(result, HYPERPARAMETERS)
        else:
            show_orbit_plot(result, HYPERPARAMETERS)
    return result




"""Browser-only playback for Colab; numerical trajectories remain unchanged."""
def create_browser_animation(result, settings):
    import html
    import json
    import numpy as np
    from IPython.display import HTML

    times = np.asarray([float(t) for t in result['t_values']], dtype=float)
    positions = np.asarray([[float(v) for v in state[:6]] for state in result['y_values']], dtype=float)
    if len(times) < 2 or positions.shape != (len(times), 6):
        raise ValueError('Animation requires at least two sampled states.')
    if not np.isfinite(times).all() or not np.isfinite(positions).all():
        raise ValueError('Animation data must be finite float64 display copies.')
    keep = np.r_[np.diff(times) > 0, True]
    times, positions = times[keep], positions[keep]
    if len(times) < 2 or times[-1] <= times[0]:
        raise ValueError('Animation requires distinct float64 display times.')
    values = result.get('proximity_values')
    if values is None:
        values = proximity_values_mp(result['y_values'][-1], result['y_values'][0])
    threshold = result.get('periodicity_threshold', mp.mpf(settings['periodicity_threshold']))
    classification = ('not assessed (incomplete)' if result['stop_reason'] != 'completed'
                      else 'periodic' if all(v < threshold for v in values.values()) else 'non-periodic')
    data = dict(times=times.tolist(), positions=positions.tolist(), colors=list(NEON_COLORS),
                label=result.get('ic_label', f"IC {settings['ic_number']}"),
                classification=classification, threshold=mp.nstr(threshold, mp.mp.dps),
                stateError=mp.nstr(values['state_error'], 2, strip_zeros=False),
                mode=settings['animation_mode'], speed=settings['animation_speed'],
                tilt=settings['view_tilt_degrees'], trail=settings['animation_trail_fraction'],
                seconds=settings['animation_seconds'], fps=min(60, float(settings['animation_fps'])),
                maxPoints=settings.get('animation_max_points', 1500),
                linewidth=settings['animation_linewidth'], triangleWidth=settings['triangle_linewidth'])
    encoded = json.dumps(data, allow_nan=False, ensure_ascii=True).replace('<', r'\u003c').replace('>', r'\u003e')
    document = r'''<!doctype html><html><head><meta charset="utf-8"><style>
html,body{margin:0;background:#000;color:#eee;font:14px system-ui,sans-serif}*{box-sizing:border-box}
#viewer{max-width:740px;margin:auto;padding:12px}h3{margin:2px 0 10px;text-align:center;font-size:18px;font-weight:500}
.info{display:flex;justify-content:space-between;gap:12px;align-items:start;min-height:70px}.box{background:#111;border:1px solid #666;border-radius:7px;padding:8px 12px;font-size:12px;line-height:1.5}
#clock{font:12px monospace;white-space:nowrap;padding-top:8px}canvas{display:block;width:100%;height:auto;touch-action:none;aspect-ratio:700/530}
.legend{display:flex;gap:20px;justify-content:center;font-size:12px;margin:3px 0 12px}.legend span{white-space:nowrap}
.controls{display:grid;grid-template-columns:1fr 1fr;gap:12px 24px;border-top:1px solid #333;padding-top:14px}
label{display:grid;grid-template-columns:95px 1fr 46px;align-items:center;gap:8px;font-size:13px}input{width:100%;accent-color:#bd72ff}output{text-align:right;font-variant-numeric:tabular-nums}
.actions{display:flex;align-items:center;gap:12px}select,button{background:#222;color:white;border:1px solid #666;border-radius:5px;padding:7px 12px;font:inherit}button{cursor:pointer}.note{color:#aaa;text-align:center;font-size:11px;margin:12px 0 0}
@media(max-width:520px){.controls{grid-template-columns:1fr}.info{gap:4px;flex-wrap:wrap}.box{font-size:11px}#clock{font-size:10px}.legend{gap:10px}}
</style></head><body><div id="viewer"><h3 id="title"></h3><div class="info"><div id="info" class="box"></div><span id="clock" role="timer"></span></div>
<canvas id="canvas" width="700" height="530" aria-label="Animated three-body trajectory"></canvas><div id="legend" class="legend"></div>
<div class="controls"><label>Speed<input id="speed" aria-label="Animation speed" type="range" min="0" max="5" step="0.05"><output id="speedValue"></output></label>
<label>Tilt from z<input id="tilt" aria-label="View tilt from z axis" type="range" min="0" max="90" step="1"><output id="tiltValue"></output></label>
<label>Trail length<input id="trail" aria-label="Orbit trail length" type="range" min="0" max="1" step="0.01"><output id="trailValue"></output></label>
<div class="actions"><label for="mode" style="display:inline">Display</label><select id="mode" aria-label="Animation display mode"><option value="orbits">Orbits</option><option value="triangle">Triangle</option></select><button id="restart" type="button">Restart</button></div></div>
<p class="note">0x = paused · right-drag to rotate · orbit trail: 0 = bodies only, 1 = full elapsed path</p></div>
<script>
'use strict';
function installRK4Player(root,d){
 const el=id=>root.querySelector('#'+id), canvas=el('canvas'),ctx=canvas.getContext('2d');
 const times=Float64Array.from(d.times), paths=d.positions, n=times.length, t0=times[0],end=times[n-1],duration=end-t0;
 let current=t0,speed=Number(d.speed),tilt=Number(d.tilt),trail=Number(d.trail),mode=d.mode,azimuth=0,dirty=true,stopped=false,atEnd=false;
 let lastDraw=-Infinity,frameId=0,drawCount=0;
 const frameMs=1000/d.fps,pointLimit=Math.max(4,Math.floor(d.maxPoints));
 let minX=Infinity,maxX=-Infinity,minY=Infinity,maxY=-Infinity;
 for(const row of paths)for(let b=0;b<3;b++){minX=Math.min(minX,row[2*b]);maxX=Math.max(maxX,row[2*b]);minY=Math.min(minY,row[2*b+1]);maxY=Math.max(maxY,row[2*b+1]);}
 const cx=(minX+maxX)/2,cy=(minY+maxY)/2,span=Math.max(maxX-minX,maxY-minY,1e-12)*1.35;
 let width=700,height=530,pixelRatio=1;
 function resize(){width=Math.max(200,canvas.clientWidth||700);height=width*530/700;pixelRatio=Math.min(window.devicePixelRatio||1,2);canvas.width=Math.round(width*pixelRatio);canvas.height=Math.round(height*pixelRatio);dirty=true;}
 function project(x,y){let a=(x-cx)*Math.cos(azimuth)-(y-cy)*Math.sin(azimuth),b=(x-cx)*Math.sin(azimuth)+(y-cy)*Math.cos(azimuth);let scale=Math.min(width,height)*.88/span;return[width/2+a*scale,height/2-b*Math.cos(tilt*Math.PI/180)*scale];}
 function bound(value,right){let lo=0,hi=n;while(lo<hi){let mid=(lo+hi)>>>1;if(times[mid]<value||(right&&times[mid]===value))lo=mid+1;else hi=mid;}return lo;}
 function positionAt(t){let i=Math.max(0,Math.min(n-2,bound(t,true)-1)),fraction=Math.max(0,Math.min(1,(t-times[i])/(times[i+1]-times[i])));return paths[i].map((v,j)=>v+(paths[i+1][j]-v)*fraction);}
 function strokePoints(points,color,lineWidth){if(points.length<2)return;ctx.beginPath();ctx.strokeStyle=color;ctx.lineWidth=lineWidth;ctx.lineJoin='round';ctx.lineCap='round';for(let i=0;i<points.length;i++){let p=project(points[i][0],points[i][1]);if(i)ctx.lineTo(p[0],p[1]);else ctx.moveTo(p[0],p[1]);}ctx.stroke();}
 function labels(){el('title').textContent='Three bodies · '+d.label+' · tilt '+tilt.toFixed(0)+'°';el('speedValue').textContent=speed.toFixed(2)+'x';el('tiltValue').textContent=tilt.toFixed(0)+'°';el('trailValue').textContent=trail.toFixed(2);el('trail').disabled=mode==='triangle';el('legend').replaceChildren();for(let b=0;b<3;b++){let item=document.createElement('span');item.style.color=d.colors[b];item.textContent=mode==='triangle'?['Side 1–2','Side 2–3','Side 3–1'][b]:'Body '+(b+1);el('legend').appendChild(item);}}
 function draw(){
  ctx.setTransform(pixelRatio,0,0,pixelRatio,0,0);ctx.fillStyle='#000';ctx.fillRect(0,0,width,height);
  const half=span/2;for(let k=-2;k<=2;k++){let v=k*span/5;strokePoints([[cx+v,cy-half],[cx+v,cy+half]],'#171717',1);strokePoints([[cx-half,cy+v],[cx+half,cy+v]],'#171717',1);}
  const currentPosition=positionAt(current);
  if(mode==='orbits'&&trail>0){
   let start=Math.max(t0,current-trail*duration),left=bound(start,true),right=bound(current,false),first=positionAt(start);
   let count=Math.max(0,right-left),budget=pointLimit-2,selected=[];
   if(count<=budget){for(let i=left;i<right;i++)selected.push(i);}else{for(let j=0;j<budget;j++)selected.push(left+Math.floor(j*(count-1)/(budget-1)));}
   for(let b=0;b<3;b++){let line=[[first[2*b],first[2*b+1]]];for(const i of selected)line.push([paths[i][2*b],paths[i][2*b+1]]);line.push([currentPosition[2*b],currentPosition[2*b+1]]);strokePoints(line,d.colors[b],d.linewidth*4/3);}
  }
  if(mode==='triangle'){for(let b=0;b<3;b++){let j=(b+1)%3;strokePoints([[currentPosition[2*b],currentPosition[2*b+1]],[currentPosition[2*j],currentPosition[2*j+1]]],d.colors[b],d.triangleWidth*4/3);}}
  for(let b=0;b<3;b++){let p=project(currentPosition[2*b],currentPosition[2*b+1]);ctx.beginPath();ctx.fillStyle=d.colors[b];ctx.arc(p[0],p[1],6,0,2*Math.PI);ctx.fill();ctx.strokeStyle='#eee';ctx.lineWidth=.7;ctx.stroke();}
  el('clock').textContent='t = '+current.toPrecision(6)+' / '+end.toPrecision(6);drawCount++;dirty=false;
 }
 function advance(seconds){if(speed<=0)return;if(atEnd){current=t0;atEnd=false;}else{current=Math.min(end,current+Math.max(0,seconds)*duration/d.seconds*speed);if(current>=end)atEnd=true;}dirty=true;}
 function tick(now){if(stopped||!root.isConnected)return;
  if(!document.hidden&&(now-lastDraw>=frameMs-.1||dirty)){if(now-lastDraw>=frameMs-.1){advance(elapsedSinceDraw(now));lastDraw=now;}if(dirty)draw();}
  if(document.hidden){lastDraw=now;lastAnimationTime=now;}
  frameId=requestAnimationFrame(tick);
 }
 let lastAnimationTime=null;
 function elapsedSinceDraw(now){let seconds=lastAnimationTime===null?0:Math.max(0,(now-lastAnimationTime)/1000);lastAnimationTime=now;return seconds;}
 function change(){labels();dirty=true;}
 el('speed').value=speed;el('tilt').value=tilt;el('trail').value=trail;el('mode').value=mode;
 el('speed').addEventListener('input',e=>{speed=Number(e.target.value);lastAnimationTime=performance.now();change();});
 el('tilt').addEventListener('input',e=>{tilt=Number(e.target.value);change();});
 el('trail').addEventListener('input',e=>{trail=Number(e.target.value);change();});
 el('mode').addEventListener('change',e=>{mode=e.target.value;change();});
 el('restart').addEventListener('click',()=>{current=t0;atEnd=false;lastAnimationTime=performance.now();dirty=true;});
 el('info').textContent='Periodicity (all four): '+d.classification+'\nThreshold: '+d.threshold+'\nFinal state_error: '+d.stateError;el('info').style.whiteSpace='pre-line';
 let dragging=null;canvas.addEventListener('contextmenu',e=>e.preventDefault());canvas.addEventListener('pointerdown',e=>{if(e.button!==2)return;dragging=[e.clientX,e.clientY];canvas.setPointerCapture(e.pointerId);});
 canvas.addEventListener('pointermove',e=>{if(!dragging)return;azimuth+=(e.clientX-dragging[0])*.008;tilt=Math.max(0,Math.min(90,tilt+(e.clientY-dragging[1])*.3));dragging=[e.clientX,e.clientY];el('tilt').value=tilt;change();});
 for(const event of ['pointerup','pointercancel'])canvas.addEventListener(event,()=>{dragging=null;});
 const visibilityChanged=()=>{lastAnimationTime=null;lastDraw=-Infinity;dirty=true;};document.addEventListener('visibilitychange',visibilityChanged);
 let observer=null;if(typeof ResizeObserver!=='undefined'){observer=new ResizeObserver(resize);observer.observe(canvas);}else window.addEventListener('resize',resize);
 resize();labels();draw();frameId=requestAnimationFrame(tick);
 const api={draw,advance,positionAt,getState:()=>({time:current,speed,tilt,trail,mode,drawCount,pointLimit}),destroy:()=>{stopped=true;cancelAnimationFrame(frameId);if(observer)observer.disconnect();else window.removeEventListener('resize',resize);document.removeEventListener('visibilitychange',visibilityChanged);}};
 root.rk4Player=api;return api;
}
installRK4Player(document.getElementById('viewer'), __RK4_DATA__);
</script></body></html>'''
    document = document.replace('__RK4_DATA__', encoded)
    # A script-enabled isolated iframe works without a Python widget callback per frame.
    return HTML('<div class="rk4-browser-output"><iframe title="Three-body animation" sandbox="allow-scripts" '
                'style="width:100%;height:880px;border:0;background:#000" srcdoc="'
                + html.escape(document, quote=True) + '"></iframe></div>')


### Build the C++ calculation backend
Run this cell after the definitions above. It installs a C++ compiler and Boost headers **only if missing**, then compiles the embedded source for the selected `dps`. The compiled program is cached for this session. Changing `dps` or the source triggers a new build.

All numerical input, trajectory samples and final errors travel as decimal text through pipes; results are not saved as files. Interrupting the integration cell also terminates its C++ child process. This implementation uses the CPU. C++ and mpmath can round slightly differently; identical long-time trajectories are not guaranteed.

The equations of motion for the C++ backend are in `CPP_SOURCE` below. Editing only the Python reference equations does not change the compiled equations. To compare the original solver, choose `INTEGRATION_BACKEND = "python"` in the settings cell.


In [ ]:
# Complete C++ source; no repository fetch is required.
CPP_SOURCE = r"""
// Self-contained adaptive RK4 kernel for the public V1.5 Colab notebook.
#include <boost/multiprecision/cpp_bin_float.hpp>
#include <array>
#include <vector>
#include <algorithm>
#include <chrono>
#include <filesystem>
#include <fstream>
#include <sstream>
#include <iostream>
#include <iomanip>
#include <stdexcept>
#include <cstdio>
#include <cmath>
#ifndef RK4_BITS
#define RK4_BITS 169
#endif
#ifndef RK4_OUTPUT_DIGITS
#define RK4_OUTPUT_DIGITS 62
#endif
using Real = boost::multiprecision::number<boost::multiprecision::cpp_bin_float<RK4_BITS, boost::multiprecision::digit_base_2>>;
using State = std::array<Real,12>;
// ================= Hyperparameters: edit here =================
struct Settings {
    Real h0,rtol,atol,safety,min_factor,max_factor;
    unsigned long long max_steps=0,progress_every=0,save_every=0;
};
struct Result { Real t=0; State y{}; unsigned long long steps=0,rejected=0; bool completed=true; std::vector<State> samples; std::vector<Real> times; };
Real mag(const Real& x){return abs(x);}
Real maximum(const Real&a,const Real&b){return a>b?a:b;}
Real minimum(const Real&a,const Real&b){return a<b?a:b;}
void positive(const Real& x,const char* label){if(!boost::multiprecision::isfinite(x)||x<=0)throw std::runtime_error(std::string(label)+" must be finite and positive");}
State initial(const Real&x,const Real&y){ State s{};s[0]=x;s[1]=y;s[2]=Real("-0.5");s[4]=Real("0.5");return s; }
State derivatives(const State&s){
    State d{};for(int k=0;k<6;++k)d[k]=s[k+6];
    for(int i=0;i<3;++i)for(int j=0;j<3;++j)if(i!=j){
        Real dx=s[2*j]-s[2*i],dy=s[2*j+1]-s[2*i+1];
        Real distance=sqrt(dx*dx+dy*dy);
        if(distance==0)throw std::runtime_error("Coincident bodies: singular gravitational acceleration");
        Real denominator=distance*distance*distance;
        d[6+2*i]+=dx/denominator;d[7+2*i]+=dy/denominator;
    }return d;
}
State rk4(const State&y,const Real&h){
    State a=derivatives(y),z{};
    for(int i=0;i<12;++i)z[i]=y[i]+h*a[i]/2;
    State b=derivatives(z);
    for(int i=0;i<12;++i)z[i]=y[i]+h*b[i]/2;
    State c=derivatives(z);
    for(int i=0;i<12;++i)z[i]=y[i]+h*c[i];
    State d=derivatives(z);
    for(int i=0;i<12;++i)z[i]=y[i]+h*(a[i]+2*b[i]+2*c[i]+d[i])/6;
    return z;
}
Result solve(const State&y0,const Real&T,const Settings&s){
    Result r;r.y=y0;r.samples.push_back(y0);r.times.push_back(Real(0));Real h=s.h0;
    while(r.t<T){
        if(r.steps>=s.max_steps){r.completed=false;break;}
        bool end=r.t+h>=T;if(end)h=T-r.t;
        if(r.t+h==r.t||r.t+h/2==r.t)throw std::runtime_error("Step size too small for compiled precision");
        State big=rk4(r.y,h),half=rk4(r.y,h/2),small=rk4(half,h/2);Real err=0;
        for(int i=0;i<12;++i)err=maximum(err,mag(small[i]-big[i])/(s.atol+s.rtol*maximum(mag(r.y[i]),mag(small[i]))));
        if(!boost::multiprecision::isfinite(err))throw std::runtime_error("Non-finite error estimate");
        if(err<=1){r.t=end?T:Real(r.t+h);r.y=small;++r.steps;
            if(r.steps%s.save_every==0){r.samples.push_back(r.y);r.times.push_back(r.t);}
            if(r.steps%s.progress_every==0)std::cout<<std::setprecision(12)<<"step "<<r.steps<<", t = "<<r.t<<" / "<<T<<", "<<100*r.t/T<<" %, h = "<<h<<", err = "<<err<<", rejected = "<<r.rejected<<std::endl;
        }else ++r.rejected;
        Real factor=err==0?s.max_factor:minimum(s.max_factor,maximum(s.min_factor,Real(s.safety*pow(err,-Real(1)/5))));h*=factor;
    }
    if(r.times.back()!=r.t){r.samples.push_back(r.y);r.times.push_back(r.t);}return r;
}
std::array<Real,3> distances(const State&s){std::array<Real,3>d{};int k=0;for(int i=0;i<3;++i)for(int j=i+1;j<3;++j){Real x=s[2*i]-s[2*j],y=s[2*i+1]-s[2*j+1];d[k++]=sqrt(x*x+y*y);}std::sort(d.begin(),d.end());return d;}
std::array<Real,4> errors(const State&y,const State&initial){
    auto a=distances(y),b=distances(initial);Real shape=0,pos=0,vel=0;
    for(int i=0;i<3;++i)shape+=(a[i]-b[i])*(a[i]-b[i]);
    for(int i=0;i<12;++i){Real diff=y[i]-initial[i];(i<6?pos:vel)+=diff*diff;}
    return {Real(sqrt(shape)),Real(sqrt(pos)),Real(sqrt(vel)),Real(sqrt(pos+vel))};
}

// Input and output use decimal text over pipes. No numerical result files.
Real read_real(){std::string token;if(!(std::cin>>token))throw std::runtime_error("Missing input value");Real x(token);if(!boost::multiprecision::isfinite(x))throw std::runtime_error("Non-finite input");return x;}
void write_state(const State&state){std::cout<<'[';for(size_t i=0;i<state.size();++i){if(i)std::cout<<',';std::cout<<'"'<<state[i]<<'"';}std::cout<<']';}
int main(){try{
    Settings s;Real T=read_real();positive(T,"end time");State y0;for(auto&v:y0)v=read_real();
    s.rtol=read_real();s.atol=read_real();s.h0=read_real();s.safety=read_real();s.min_factor=read_real();s.max_factor=read_real();
    if(!(std::cin>>s.max_steps>>s.progress_every>>s.save_every)||!s.max_steps||!s.progress_every||!s.save_every)throw std::runtime_error("Invalid step counters");
    positive(s.rtol,"rtol");positive(s.atol,"atol");positive(s.h0,"h0");
    if(!(s.safety>0&&s.safety<1&&s.min_factor>0&&s.min_factor<1&&s.max_factor>1))throw std::runtime_error("Invalid step adjustment factors");
    auto start=std::chrono::steady_clock::now();Result r=solve(y0,T,s);auto e=errors(r.y,y0);double seconds=std::chrono::duration<double>(std::chrono::steady_clock::now()-start).count();
    std::cout<<std::setprecision(RK4_OUTPUT_DIGITS)<<"{\"event\":\"result\",\"t\":\""<<r.t<<"\",\"steps\":"<<r.steps<<",\"rejected\":"<<r.rejected<<",\"stop_reason\":\""<<(r.completed?"completed":"max_steps")<<"\",\"integration_seconds\":"<<std::setprecision(17)<<seconds<<std::setprecision(RK4_OUTPUT_DIGITS)<<",\"precision_bits\":"<<RK4_BITS<<",\"y\":";
    write_state(r.y);std::cout<<",\"t_values\":[";for(size_t i=0;i<r.times.size();++i){if(i)std::cout<<',';std::cout<<'"'<<r.times[i]<<'"';}std::cout<<"],\"y_values\":[";
    for(size_t i=0;i<r.samples.size();++i){if(i)std::cout<<',';write_state(r.samples[i]);}std::cout<<"],\"proximity_values\":{";
    const char*names[]={"shape_error","position_error","velocity_error","state_error"};for(int i=0;i<4;++i){if(i)std::cout<<',';std::cout<<'"'<<names[i]<<"\":\""<<e[i]<<'"';}std::cout<<"}}"<<std::endl;return r.completed?0:2;
}catch(const std::exception&e){std::cerr<<"C++ error: "<<e.what()<<std::endl;return 1;}}

"""

"""Self-contained C++ compilation/pipe bridge embedded in the public notebook."""
import hashlib
import json
import os
import platform
import shutil
import subprocess
import tempfile
from pathlib import Path

_RK4_CPP_BUILDS = globals().get('_RK4_CPP_BUILDS', {})


def prepare_cpp_backend(dps):
    """Compile once per source/precision combination; only build files touch disk."""
    if type(dps) is not int or dps < 1:
        raise ValueError('dps must be a positive integer.')
    with mp.workdps(dps):
        bits = mp.mp.prec
    output_digits = dps + 12
    key = hashlib.sha256((CPP_SOURCE + f'\n{bits}:{output_digits}').encode()).hexdigest()
    if key in _RK4_CPP_BUILDS:
        return _RK4_CPP_BUILDS[key][1]

    include_dirs = [Path('/usr/include'), Path('/opt/homebrew/include'), Path('/usr/local/include')]
    boost_dirs = [p for p in include_dirs if (p / 'boost/multiprecision/cpp_bin_float.hpp').exists()]
    compiler = shutil.which('g++') or shutil.which('clang++')
    if not compiler or not boost_dirs:
        # Managed Colab sessions normally permit apt installation. No shell is used.
        if platform.system() != 'Linux' or shutil.which('apt-get') is None:
            raise RuntimeError('A C++17 compiler and Boost headers are required. '
                               'Install them or set INTEGRATION_BACKEND = "python".')
        print('Installing the C++ compiler and Boost headers for this session...', flush=True)
        for command in (['apt-get', 'update', '-qq'],
                        ['apt-get', 'install', '-y', '-qq', 'g++', 'libboost-dev']):
            completed = subprocess.run(command, text=True, capture_output=True)
            if completed.returncode:
                raise RuntimeError('C++ dependency installation failed:\n'
                                   + (completed.stdout + completed.stderr)[-8000:])
        compiler = shutil.which('g++')
        boost_dirs = [Path('/usr/include')]
    directory = tempfile.TemporaryDirectory(prefix='rk4_cpp_')
    source = Path(directory.name) / 'rk4.cpp'
    executable = Path(directory.name) / 'rk4'
    source.write_text(CPP_SOURCE, encoding='utf-8')
    command = [compiler, '-O3', '-std=c++17', f'-DRK4_BITS={bits}',
               f'-DRK4_OUTPUT_DIGITS={output_digits}']
    for include in boost_dirs:
        command.extend(['-I', str(include)])
    if platform.system() == 'Darwin':
        sdk = subprocess.check_output(['xcrun', '--show-sdk-path'], text=True).strip()
        command.extend(['-isystem', str(Path(sdk) / 'usr/include/c++/v1')])
    command.extend([str(source), '-o', str(executable)])
    print(f'Compiling C++ RK4 for dps={dps} ({bits} working bits)...', flush=True)
    completed = subprocess.run(command, text=True, capture_output=True)
    if completed.returncode:
        directory.cleanup()
        raise RuntimeError('C++ compilation failed:\n' + completed.stderr[-8000:])
    _RK4_CPP_BUILDS[key] = (directory, executable)
    print('C++ backend ready. Trajectory results stay in memory.', flush=True)
    return executable


def solve_adaptive_rk4_cpp(f, y0, T, *, rtol, atol, h0, safety,
                           min_factor, max_factor, max_steps,
                           progress_every, save_every):
    """Run the specialized three-body C++ kernel and decode its decimal results."""
    if f is not derivatives_mp:
        raise ValueError('This C++ backend implements derivatives_mp for the three-body problem.')
    if len(y0) != 12:
        raise ValueError('The initial state must have 12 components.')
    executable = prepare_cpp_backend(mp.mp.dps)
    decimal = lambda value: mp.nstr(mp.mpf(value), mp.mp.dps + 12)
    data = [T, *y0, rtol, atol, h0, safety, min_factor, max_factor]
    if not all(mp.isfinite(mp.mpf(value)) for value in data):
        raise ValueError('All integration inputs must be finite.')
    for value in (max_steps, progress_every, save_every):
        if type(value) is not int or value < 1:
            raise ValueError('Step limits and sampling/progress intervals must be positive integers.')
    payload = ' '.join(map(decimal, data)) + f' {max_steps} {progress_every} {save_every}\n'
    process = subprocess.Popen([str(executable)], stdin=subprocess.PIPE,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    raw = None
    diagnostics = []
    try:
        process.stdin.write(payload)
        process.stdin.close()
        for line in process.stdout:
            if line.startswith('{"event":"result"'):
                raw = json.loads(line)
            else:
                print(line, end='', flush=True)
                diagnostics.append(line)
                diagnostics = diagnostics[-10:]
        code = process.wait()
        if code not in (0, 2) or raw is None:
            raise RuntimeError(f'C++ integration failed (exit {code}).\n' + ''.join(diagnostics))
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise
    finally:
        process.stdout.close()
        if process.stdin is not None and not process.stdin.closed:
            process.stdin.close()
    raw.pop('event', None)
    raw['t'] = mp.mpf(raw['t'])
    raw['y'] = list(map(mp.mpf, raw['y']))
    raw['t_values'] = list(map(mp.mpf, raw['t_values']))
    raw['y_values'] = [list(map(mp.mpf, row)) for row in raw['y_values']]
    raw['proximity_values'] = {name: mp.mpf(value) for name, value in raw['proximity_values'].items()}
    raw['backend'] = 'cpp'
    return raw

if INTEGRATION_BACKEND == "cpp":
    prepare_cpp_backend(HYPERPARAMETERS["dps"])


## 5. Integrate and print the proximity values

Progress reports show accepted steps, integrated time, percentage, step size, error estimate and rejected steps. Classification compares the final state with the initial state using the selected numerical threshold; it is a numerical return test, not a proof of periodicity.


In [ ]:
# Keep the result in memory so the display can be rerun without reintegration.
result = main(show_plots=False)


## 6. Display the orbit plot and animation

The default **browser** renderer plays the animation directly in your browser. Display samples are sent once; Python and the network are not used for each animation frame. The static orbit plot stays interactive through Matplotlib.

Select **Orbits / Triangle**, change **Speed** (0 = paused), **Tilt from z** (0–90°) and **Trail length** (0–1), or press **Restart**. Right-drag on the animation to rotate the view. The black background, neon body/side colours and periodicity information are retained. The animation is an orthographic view of the planar motion; the integration and final error tests are unchanged.

`animation_max_points` limits the trail vertices drawn per body/frame (default 1500), keeping full trails responsive. All numerical samples remain intact, and the body positions use the complete saved time series. Increase this display limit if you need more trail detail. Triangle mode does not process hidden orbit trails. Playback interpolates between saved samples; it cannot reconstruct motion omitted by `save_every`.

Set `animation_renderer = "matplotlib"` for the legacy animation. FPS is capped at 60 in the browser renderer. Line widths still use `animation_linewidth` and `triangle_linewidth`.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

# Close earlier figures if this display cell is rerun. Keep the integration result.
if "animation_view" in globals() and animation_view is not None:
    animation_view.animation.event_source.stop()
    plt.close(animation_view.fig)
if "orbit_fig" in globals():
    plt.close(orbit_fig)
animation_view = None

# Explicit canvas display avoids duplicate inline figures in the notebook.
with plt.ioff():
    orbit_fig = create_orbit_plot(result, HYPERPARAMETERS)
display(orbit_fig.canvas)
orbit_fig.canvas.draw_idle()

if HYPERPARAMETERS["animate"] and len(result["t_values"]) >= 2:
    if HYPERPARAMETERS.get("animation_renderer", "browser") == "browser":
        # Send display samples once; playback and controls stay in the browser.
        browser_animation_view = create_browser_animation(result, HYPERPARAMETERS)
        display(browser_animation_view)
    else:
        with plt.ioff():
            animation_view = OrbitAnimation(result, HYPERPARAMETERS)
        display(animation_view.fig.canvas)
        animation_view.last_tick = perf_counter()
        animation_view.fig.canvas.draw_idle()


## Sharing this notebook

This notebook is hosted in the public [GitHub repository](https://github.com/Mattias-XD/three-body-problem). [Open it in Google Colab](https://colab.research.google.com/github/Mattias-XD/three-body-problem/blob/main/code/rk4_mpmath_base_V1.5_Colab.ipynb) and run the cells in order in your own runtime. You may need to sign in to Google to execute cells. Use **File → Save a copy in Drive** to keep your own editable copy. The catalogue is downloaded from the authors’ public website; Google Drive access is not needed for the simulation.
